<a href="https://colab.research.google.com/github/SURENAANERUS/Projects/blob/main/Copy_of_model19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive') # import the data from google Drive

In [ ]:
import numpy as np
import torch

b = np.load('/content/drive/MyDrive/chess_dataset_0 (3).npz') # load the data


In [ ]:
xNp = b['X']
yNp = b['y']
idsNp = b['game_ids']

x = torch.from_numpy(xNp)
y = torch.from_numpy(yNp)
ids = torch.from_numpy(idsNp)


# get rid of the useless matrices

x = x[:,:16]
print(x.shape)

# free ram

import gc

del xNp, yNp, idsNp  # delete your original numpy arrays
gc.collect()    # force garbage collection

In [ ]:

use_cuda = torch.cuda.is_available()
device = "cuda" if use_cuda else "cpu"
print("device:", device)
kwargs = {"num_workers":0,"pin_memory": True, "shuffle":True} if use_cuda else{"shuffle":True}
kwargsTest = {"num_workers":0,"pin_memory": True, "shuffle":False} if use_cuda else{"shuffle":False}

trainRatio = 0.7

trainLen = int(len(x) * trainRatio)
xTrain,xTest = x[:trainLen].float(), x[trainLen:].float()
yTrain, yTest = y[:trainLen].float(), y[trainLen:].float()
trainData = torch.utils.data.TensorDataset(xTrain,yTrain)
testData = torch.utils.data.TensorDataset(xTest,yTest)


xTrain = xTrain.to(device)
yTrain = yTrain.to(device)

xTest = xTest.to(device)
yTest = yTest.to(device)




#print(x.shape, y.shape, ids.shape) # these are  8024029 x 18 x 8 x 8, so 8024029 instances, of 18 tensors, 8x8 boards.
bs = 2048 # let's try 64 for now
trainDL = torch.utils.data.DataLoader(trainData,bs,**kwargs)
testDL = torch.utils.data.DataLoader(testData,bs,**kwargsTest)




In [ ]:
import torch.nn as nn
# now it's time to create the network itself.
# we start with B x 16 x 8 x 8
class Model(nn.Module):
  def __init__(self):
    super().__init__()
    self.reLu = nn.ReLU()
    self.bn0 = nn.BatchNorm2d(16)

    self.conv1 = nn.Conv2d(16,32,3,1,padding="same") # now its 32 x 8 x 8
    self.bn1 = nn.BatchNorm2d(32)
    self.pool1 = nn.MaxPool2d((2,2)) # now its 32 x 4 x 4

    self.conv2 = nn.Conv2d(32,64,3,1,padding="same") # now its  64 x 4 x 4
    self.bn2 = nn.BatchNorm2d(64)
    self.pool2 = nn.MaxPool2d((2,2)) # now its 64 x 2 x 2

    self.conv3 = nn.Conv2d(64,128,3,1,padding="same") # now its 128 x 4 x 4
    self.bn3 = nn.BatchNorm2d(128)
    self.pool3 =


    self.ln = nn.Linear(4,1)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    # x comes as B x 16 x 8 x 8
    #x = self.bn0(x)
    #print("enter 1 block")
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.reLu(x)
    x = self.pool1(x)

    #print("enter 2 block")

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.reLu(x)
    x = self.pool2(x)
    #print("enter 3 block")
    x = self.conv3(x)
    x = self.bn3(x)
    x = self.reLu(x)
    #print("enter 4 block")
    x = self.deconv1(x)
    x = self.bn4(x)
    x = self.reLu(x)
    #print("enter 5 block")
    x = self.deconv2(x)
    x = self.bn5(x)
    x = self.reLu(x)
    #print("enter 6 block")
    x = self.deconv3(x)
    x = self.bn6(x)
    x = self.reLu(x)
    #print("enter 7 block")
    x = self.deconv4(x)
    x = self.reLu(x)
    #print("enter 8 block")
    #print("SHAPE:", x.shape) # SHAPE: torch.Size([64, 1, 2, 2])

    x = x.view(-1,1 * 2 * 2) # now its 64 x 4
    #print("NEW SHAPE: ", x.shape)
    x = self.ln(x)
    x = x.squeeze(1)
    #print("final x shape: ", x.shape)
    return x

model = Model().to(device)
# now onto the train and test loops



In [ ]:
import torch.nn.functional as F
def train_epoch(
    model: torch.nn.Module,
    device: torch.device,
    trainLoader: torch.utils.data.DataLoader,
    opt: torch.optim.Optimizer,
    epoch:int,
    log_interval: int,
) -> None:
  model.train()
  for batchIdx, (data, target) in enumerate(trainLoader):
    opt.zero_grad()
    output = model(data)
    loss = F.binary_cross_entropy_with_logits(output,target)

    loss.backward()
    opt.step()

    if batchIdx % log_interval == 0:
      print(f"Train Epoch: {epoch}, training loss: {loss.item():.6f}")

def test(
  model: torch.nn.Module,
  device: torch.device,
  testLoader: torch.utils.data.DataLoader,
  epoch:int,
) -> None:
  model.eval()
  testLoss = 0
  nCorrect = 0
  nTotal = 0
  with torch.no_grad():
    for data,target in testLoader:

      output = model(data)
      testLoss += F.binary_cross_entropy_with_logits(output,target)
      probs = torch.sigmoid(output) # we need probabilities, not logits
      preds = (probs > 0.5).float()

      nCorrect += (preds == target).sum().item()
      nTotal += len(target)
    testLoss /= nTotal
    accuracy = nCorrect / nTotal

    print(f"Test loss: {testLoss:.4f}, Accuracy: {accuracy}")
      # now preds should be B x 1, with True False True false


In [ ]:
# Let it rip
EPOCHS = 10
optimizer = torch.optim.Adam(model.parameters()) # defcault lr is 0.01 IIRC

for epoch in range(EPOCHS):
  train_epoch(model,device,trainDL,optimizer,epoch,10000)
  print("train done!")
  test(model,device,testDL,epoch)
  print("test done! ")

